[ссылка на датасет](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

In [41]:
#импорты
import pandas as pd
import sklearn as sk
import numpy as np
import joblib


from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [26]:
#загрузка данных
data = pd.read_csv('/Users/danilaskiba/git_repository/pmrd/data/raw/creditcard.csv')

data.info()

<class 'pandas.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     284807 non-nu

In [12]:
#разбиение на обучающую/тестовую выборки
X = data.drop(columns=['Class'])
Y = data['Class']

X_train, X_test, Y_train, Y_test = sk.model_selection.train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

In [15]:
#искусственные пропуски
for sample in (X_train, X_test):
    for column in ['V1', 'V2']:
        rows = np.random.choice(
            sample.index,
            size=round(0.1 * len(sample)),
            replace=False
        )
        sample.loc[rows,column] = np.nan

In [ ]:
num_columns = X_train.columns.drop('Amount') #Все числовые признаки, исключая Amount - обрабатывает кастомынм трансоформером
categorical_columns = X_train.select_dtypes(
    include = ['object', 'category', 'string']
).columns.tolist()

#построение пайплайнов
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])



In [ ]:
#Кастомный трансформер для масштабирования колонки Amount через MinMaxScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler

class AmountMinMaxTransformer(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        self.scaler_ = MinMaxScaler().fit(X)
        return self

    def transform(self, X):
        return self.scaler_.transform(X)
    


In [35]:

#Шаг в пайплайне
amount_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", AmountMinMaxTransformer()),
])

#Сборка пайплайна
preprocessing = ColumnTransformer([
    ("numeric", num_pipeline, num_columns),
    ("categorical", categorical_pipeline, categorical_columns),
    ("amount", amount_pipeline, ['Amount'])
])


In [39]:
#Логистическая регрессия

pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

search = GridSearchCV(
    pipeline,
    param_grid={"classifier__C": [0.1, 1.0, 10.0]},
    scoring="average_precision",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
)
search.fit(X_train, Y_train)

model = search.best_estimator_
scores = model.predict_proba(X_test)[:, 1]
predictions = model.predict(X_test)

print("Лучшие параметры:", search.best_params_)
pd.Series({
    "ROC-AUC": roc_auc_score(Y_test, scores),
    "Average Precision": average_precision_score(Y_test, scores),
    "Accuracy": accuracy_score(Y_test, predictions),
}).round(4)

Лучшие параметры: {'classifier__C': 0.1}


ROC-AUC              0.9629
Average Precision    0.7427
Accuracy             0.9991
dtype: float64

In [44]:
#Сохранение модели

model_path = "/Users/danilaskiba/git_repository/pmrd/models/fraud_pipeline.joblib"
joblib.dump(model, model_path)

['/Users/danilaskiba/git_repository/pmrd/models/fraud_pipeline.joblib']